# Instructor Solution - HW2 QFT / inverse-QFT Hardware Recovery

This notebook shows the instructor/reference workflow. The hidden file `hw2_reference.py` is the source of truth for reference generation and demo grading.

In [ ]:
%pip -q install qiskit qiskit-aer qiskit-ibm-runtime matplotlib

In [ ]:
import json
from pathlib import Path
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

from hw2_reference import (
    generate_config,
    build_reference_circuit,
    simulate_reference_counts,
    expected_display_bitstring,
    expected_hardware_display_bitstring,
    expected_logical_q_string,
    reference_answers,
    validate_answers,
)

In [ ]:
STUDENT_ID = "demo_student"
ASSIGNMENT_ID = "HW2"
SHOTS = 2048

config = generate_config(STUDENT_ID, ASSIGNMENT_ID)
print(json.dumps(config, indent=2))

## Reference circuit and expected simulator result

In [ ]:
qc_ref = build_reference_circuit(config)
qc_ref.draw("mpl")

In [ ]:
expected_display = expected_display_bitstring(config)
expected_logical = expected_logical_q_string(config["input_bits_q0_to_qn"])
counts_ref = simulate_reference_counts(config, shots=SHOTS)

print("Expected logical q[n-1]...q[0]:", expected_logical)
print("Expected displayed count string c[n-1]...c[0]:", expected_display)
print("Reference counts:", counts_ref)
print("Dominant:", max(counts_ref, key=counts_ref.get))
plot_histogram(counts_ref, title="Instructor reference simulator result")

## Transpiler statistics

In [ ]:
sim = AerSimulator(seed_simulator=config["seed"] % (2**32 - 1))
tqc_ref = transpile(qc_ref, sim, seed_transpiler=config["seed"] % (2**32 - 1), optimization_level=0)
print("Original depth:", qc_ref.depth())
print("Transpiled simulator depth:", tqc_ref.depth())
print("Operation counts:", dict(tqc_ref.count_ops()))

## Full generated reference answer

This is the instructor-side expected answer object. A grader can regenerate this from the student ID rather than storing per-student answer keys.

In [ ]:
ref = reference_answers(STUDENT_ID, ASSIGNMENT_ID, shots=SHOTS)
print(json.dumps(ref, indent=2))

## Optional: validate a submitted answers.json

Place a student's `answers.json` in the same directory and run this cell.

In [ ]:
# if Path('answers.json').exists():
#     with open('answers.json') as f:
#         submitted = json.load(f)
#     result = validate_answers(submitted, STUDENT_ID, ASSIGNMENT_ID)
#     print(json.dumps(result, indent=2))
# else:
#     print('No answers.json found in this directory.')

## Optional hardware reference notes

The hardware section is intentionally not part of the exact autograded core. Real hardware counts should be graded with tolerance/interpretation rather than exact matching. The instructor can still regenerate the expected hardware bitstring for the small hardware subset.

In [ ]:
print("Hardware subset qubits:", config["hardware_num_qubits"])
print("Hardware input bits q0...qn:", config["hardware_input_bits_q0_to_qn"])
print("Expected hardware displayed bitstring:", expected_hardware_display_bitstring(config))